# QQQ와 예금의 잔액 흐름 비교

세 시작연도와 비교 조건을 입력한 뒤 아래 셀을 실행하세요. QQQ와 예금의 월별 잔액을 비교하고 그래프와 상세 CSV를 생성합니다.

In [ ]:
# @title 비교 조건을 입력하고 실행하세요
최저성과_시작연도 = 2002  # @param {type:"integer"}
중간성과_시작연도 = 2010  # @param {type:"integer"}
높은성과_시작연도 = 2016  # @param {type:"integer"}

초기준비금_억원 = 1.0  # @param {type:"number"}
월인출액_만원 = 100  # @param {type:"number"}
예금금리_퍼센트 = 3.0  # @param {type:"number"}
비교기간_년 = 10  # @param {type:"integer"}

"""QQQ와 정기예금의 월별 인출 후 잔액 경로를 세 사례로 비교한다.

QQQ와 원/달러 환율 데이터를 직접 내려받아 월별 CSV와 그래프를 생성한다.
두 상품에 동일한 초기 준비금과 월 인출액을 적용하며, 예금금리는 월 복리로
환산한다. 세금과 거래·환전 수수료는 반영하지 않는다.
"""

import os
from pathlib import Path
import subprocess
import sys
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.axes import Axes
from matplotlib import font_manager
import pandas as pd


def is_colab_runtime() -> bool:
    """현재 코드가 Google Colab에서 실행 중인지 확인한다."""
    return bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()


IS_COLAB = is_colab_runtime()

try:
    import yfinance as yf
except ImportError:
    if IS_COLAB:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "yfinance"]
        )
        import yfinance as yf
    else:
        raise


SCRIPT_DIR = (
    Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
)
DEFAULT_OUTPUT_DIR = Path("/content/output") if IS_COLAB else SCRIPT_DIR / "output"
DEFAULT_DETAIL_OUTPUT = (
    DEFAULT_OUTPUT_DIR / "qqq_vs_deposit_balance_flow_monthly_detail.csv"
)
DEFAULT_OUTPUT = DEFAULT_OUTPUT_DIR / "qqq_vs_deposit_balance_flow.png"

WON_PER_EOK = 100_000_000
WON_PER_MANWON = 10_000
MONTHS_PER_YEAR = 12
MIN_START_YEAR = 2000

START_YEARS = tuple(
    int(year)
    for year in (최저성과_시작연도, 중간성과_시작연도, 높은성과_시작연도)
)
CASE_LABELS = ("최저 사례", "중간 사례", "최고 사례")
CASES = tuple(zip(START_YEARS, CASE_LABELS))

INITIAL_RESERVE = float(초기준비금_억원) * WON_PER_EOK
MONTHLY_WITHDRAWAL = float(월인출액_만원) * WON_PER_MANWON
ANNUAL_DEPOSIT_RATE = float(예금금리_퍼센트) / 100
COMPARISON_YEARS = int(비교기간_년)
MONTHS = COMPARISON_YEARS * MONTHS_PER_YEAR
DEPOSIT_LABEL = f"예금(연 {ANNUAL_DEPOSIT_RATE * 100:g}%)"

QQQ_COLOR = "#2F7DD3"
DEPOSIT_COLOR = "#8D8B85"
DEPLETION_COLOR = "#F06432"
REFERENCE_COLOR = "#F59E0B"
BACKGROUND_COLOR = "#FFFFFF"
GRID_COLOR = "#DEDCD6"
TEXT_COLOR = "#0B0B0B"
SECONDARY_TEXT_COLOR = "#666666"
TICK_COLOR = "#777777"
FOOTNOTE_COLOR = "#777777"
FONT_CANDIDATES = (
    "Pretendard",
    "Apple SD Gothic Neo",
    "Noto Sans CJK KR",
    "Malgun Gothic",
)
COLAB_FONT_PATH = Path("/content/.fonts/Pretendard-Regular.otf")
COLAB_FONT_URL = (
    "https://raw.githubusercontent.com/orioncactus/pretendard/main/"
    "packages/pretendard/dist/public/static/Pretendard-Regular.otf"
)


def validate_parameters() -> None:
    """사용자가 입력한 비교 조건을 검증한다."""
    if len(set(START_YEARS)) != len(START_YEARS):
        raise ValueError("세 시작연도는 서로 달라야 합니다.")
    if min(START_YEARS) < MIN_START_YEAR:
        raise ValueError(
            f"QQQ 데이터 범위를 고려해 시작연도는 {MIN_START_YEAR}년 이후로 입력하세요."
        )
    if INITIAL_RESERVE <= 0:
        raise ValueError("초기 준비금은 0보다 커야 합니다.")
    if MONTHLY_WITHDRAWAL <= 0:
        raise ValueError("월 인출액은 0보다 커야 합니다.")
    if ANNUAL_DEPOSIT_RATE < 0:
        raise ValueError("예금금리는 0% 이상이어야 합니다.")
    if COMPARISON_YEARS <= 0:
        raise ValueError("비교기간은 1년 이상이어야 합니다.")


validate_parameters()


# =============================================================================
# 1. 실행 환경과 한글 글꼴
# =============================================================================


def configure_korean_font() -> None:
    """사용 가능한 한글 글꼴을 Matplotlib에 설정한다."""
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font_name = next(
        (name for name in FONT_CANDIDATES if name in installed),
        None,
    )

    if font_name is None and IS_COLAB:
        COLAB_FONT_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not COLAB_FONT_PATH.exists():
            urlretrieve(COLAB_FONT_URL, COLAB_FONT_PATH)
        font_manager.fontManager.addfont(COLAB_FONT_PATH)
        font_name = font_manager.FontProperties(
            fname=COLAB_FONT_PATH
        ).get_name()

    if font_name is not None:
        plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False


# =============================================================================
# 2. 시장 데이터 수집
# =============================================================================


def download_monthly_qqq(start: str, end: str) -> pd.Series:
    """Yahoo Finance에서 QQQ 월말 수정주가를 내려받는다."""
    raw_prices = yf.download(
        "QQQ",
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        actions=False,
    )
    if raw_prices is None or raw_prices.empty:
        raise RuntimeError("QQQ 가격 데이터를 내려받지 못했습니다.")
    adjusted_close = raw_prices["Adj Close"]
    if isinstance(adjusted_close, pd.DataFrame):
        adjusted_close = (
            adjusted_close["QQQ"]
            if "QQQ" in adjusted_close
            else adjusted_close.iloc[:, 0]
        )
    return adjusted_close.dropna().astype(float).resample("ME").last()


def download_monthly_usdkrw(start: str, end: str) -> pd.Series:
    """FRED에서 월말 원/달러 환율을 내려받는다."""
    fred_url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id=DEXKOUS&cosd={start}&coed={end}"
    )
    raw_fx = pd.read_csv(fred_url)
    date_column = "observation_date" if "observation_date" in raw_fx else "DATE"
    raw_fx[date_column] = pd.to_datetime(raw_fx[date_column])
    fx_values = pd.to_numeric(raw_fx["DEXKOUS"], errors="coerce")
    daily_fx = pd.Series(
        fx_values.to_numpy(), index=raw_fx[date_column], name="원달러환율"
    ).dropna()
    return daily_fx.resample("ME").last()


def download_monthly_market_data() -> tuple[pd.Series, pd.Series]:
    """세 사례 전체를 계산하는 데 필요한 시장 데이터를 준비한다."""
    first_year = min(START_YEARS)
    last_year = max(START_YEARS)
    start = f"{first_year - 1}-12-01"
    end = f"{last_year + COMPARISON_YEARS}-12-31"
    yahoo_end = f"{last_year + COMPARISON_YEARS + 1}-01-01"  # Yahoo 종료일은 포함되지 않는다.
    return download_monthly_qqq(start, yahoo_end), download_monthly_usdkrw(
        start, end
    )


# =============================================================================
# 3. QQQ와 예금 백테스트
# =============================================================================


def prepare_case_market_data(
    monthly_prices: pd.Series, monthly_fx: pd.Series, start_year: int
) -> pd.DataFrame:
    """한 사례에 필요한 가격·환율을 같은 월말 인덱스로 정렬한다."""
    first_month = pd.Timestamp(start_year, 1, 31)
    previous_month = first_month - pd.offsets.MonthEnd(1)
    final_month = first_month + pd.offsets.MonthEnd(MONTHS - 1)
    market = pd.concat(
        [
            monthly_prices.rename("QQQ수정주가"),
            monthly_fx.rename("원달러환율"),
        ],
        axis=1,
    ).loc[previous_month:final_month]
    expected_count = MONTHS + 1
    if len(market) != expected_count or market.isna().any().any():
        raise RuntimeError(
            f"{start_year}년 시장 데이터가 부족하거나 월말 인덱스가 일치하지 않습니다 "
            f"({expected_count}개월 필요)."
        )
    market["월수익률"] = market["QQQ수정주가"].pct_change()
    return market


def simulate_qqq_case(
    monthly_prices: pd.Series, monthly_fx: pd.Series, start_year: int
) -> pd.DataFrame:
    """한 시작 연도의 QQQ 월별 인출 과정과 잔액을 계산한다."""
    market = prepare_case_market_data(monthly_prices, monthly_fx, start_year)

    balance_usd = INITIAL_RESERVE / float(market["원달러환율"].iloc[0])
    rows: list[dict[str, object]] = []
    for date, values in market.iloc[1:].iterrows():
        monthly_return = float(values["월수익률"])
        fx_rate = float(values["원달러환율"])
        before_withdrawal_usd = balance_usd * (1 + monthly_return)
        requested_withdrawal_usd = MONTHLY_WITHDRAWAL / fx_rate
        withdrawal_usd = min(
            requested_withdrawal_usd, max(before_withdrawal_usd, 0.0)
        )
        withdrawal_krw = withdrawal_usd * fx_rate
        balance_usd = max(before_withdrawal_usd - withdrawal_usd, 0.0)
        rows.append(
            {
                "시작연도": start_year,
                "날짜": date,
                "원달러환율": fx_rate,
                "월수익률": monthly_return,
                "인출전잔액(달러)": before_withdrawal_usd,
                "요청인출액(원)": MONTHLY_WITHDRAWAL,
                "실제인출액(달러)": withdrawal_usd,
                "실제인출액(원)": withdrawal_krw,
                "월말잔액(달러)": balance_usd,
                "월말잔액(원)": balance_usd * fx_rate,
            }
        )
        if balance_usd <= 0:
            break
    return pd.DataFrame(rows)


def create_detail_csv(output_path: Path) -> Path:
    """시장 데이터를 받아 세 사례의 월별 상세 CSV를 생성한다."""
    monthly_prices, monthly_fx = download_monthly_market_data()
    detail = pd.concat(
        [
            simulate_qqq_case(monthly_prices, monthly_fx, year)
            for year in START_YEARS
        ],
        ignore_index=True,
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    detail.to_csv(output_path, index=False, encoding="utf-8-sig")
    return output_path


def extract_balance_path(detail: pd.DataFrame, start_year: int) -> list[float]:
    """상세 데이터에서 한 사례의 고정 길이 잔액 경로를 만든다."""
    rows = detail.loc[detail["시작연도"] == start_year].sort_values("날짜")
    if rows.empty:
        raise ValueError(f"상세 CSV에 {start_year}년 데이터가 없습니다.")

    balances = rows["월말잔액(원)"].astype(float).tolist()[:MONTHS]
    if len(balances) < MONTHS and balances[-1] > 0:
        raise ValueError(
            f"{start_year}년 데이터가 {len(balances)}개월에서 끝났지만 "
            "마지막 잔액이 0원이 아닙니다."
        )
    balances.extend([0.0] * (MONTHS - len(balances)))
    return [float(INITIAL_RESERVE), *balances]


def load_qqq_paths(detail_csv: Path) -> dict[int, list[float]]:
    """상세 CSV에서 각 사례의 월말 원화 잔액 경로를 읽는다."""
    if not detail_csv.exists():
        raise FileNotFoundError(f"월별 상세 CSV가 없습니다: {detail_csv}")

    detail = pd.read_csv(detail_csv)
    required_columns = {"시작연도", "날짜", "월말잔액(원)"}
    missing_columns = required_columns.difference(detail.columns)
    if missing_columns:
        raise ValueError(
            f"상세 CSV에 필요한 열이 없습니다: {sorted(missing_columns)}"
        )

    return {year: extract_balance_path(detail, year) for year in START_YEARS}


def calculate_deposit_path() -> list[float]:
    """정기예금에서 매월 말 인출한 뒤의 잔액 경로를 계산한다."""
    monthly_rate = (1 + ANNUAL_DEPOSIT_RATE) ** (1 / MONTHS_PER_YEAR) - 1
    balance = float(INITIAL_RESERVE)
    path = [balance]
    for _ in range(MONTHS):
        balance = max(balance * (1 + monthly_rate) - MONTHLY_WITHDRAWAL, 0.0)
        path.append(balance)
    return path


def first_depletion_month(path: list[float]) -> int | None:
    """최초 고갈 월을 반환하고 고갈되지 않으면 None을 반환한다."""
    return next(
        (month for month, balance in enumerate(path[1:], 1) if balance <= 0),
        None,
    )


# =============================================================================
# 4. 그래프 작성
# =============================================================================


def format_balance(amount: float) -> str:
    """원화 잔액을 억원·만원 단위로 표시한다."""
    rounded_manwon = int(amount / WON_PER_MANWON + 0.5)
    if rounded_manwon == 0:
        return "0원"
    eok, manwon = divmod(rounded_manwon, WON_PER_EOK // WON_PER_MANWON)
    if eok and manwon:
        return f"{eok}억 {manwon:,}만원"
    if eok:
        return f"{eok}억원"
    return f"{manwon:,}만원"


def format_eok_axis(value: float, _: float) -> str:
    """억원 축 눈금에서 불필요한 소수점 0을 제거한다."""
    return f"{value / WON_PER_EOK:.1f}".rstrip("0").rstrip(".")


def format_duration(month: int) -> str:
    """개월 수를 읽기 쉬운 연·개월 문자열로 변환한다."""
    years, remaining_months = divmod(month, MONTHS_PER_YEAR)
    if remaining_months == 0:
        return f"{years}년"
    if years == 0:
        return f"{remaining_months}개월"
    return f"{years}년 {remaining_months}개월"


def add_path_annotation(
    ax: Axes,
    path: list[float],
    color: str,
    start_year: int,
    depletion_label_offset: int,
) -> None:
    """고갈 지점 또는 10년 후 잔액을 선 위에 표시한다."""
    depletion_month = first_depletion_month(path)
    if depletion_month is not None:
        depletion_year = start_year + depletion_month / MONTHS_PER_YEAR
        ax.scatter(depletion_year, 0, s=65, color=DEPLETION_COLOR, zorder=5)
        ax.annotate(
            f"{format_duration(depletion_month)} 후 고갈",
            xy=(depletion_year, 0),
            xytext=(0, depletion_label_offset),
            textcoords="offset points",
            ha="center",
            color=color,
            fontsize=13,
            fontweight="bold",
            bbox={
                "boxstyle": "round,pad=0.2",
                "facecolor": BACKGROUND_COLOR,
                "edgecolor": "none",
                "alpha": 0.82,
            },
        )
    else:
        ending_year = start_year + MONTHS / MONTHS_PER_YEAR
        ax.scatter(ending_year, path[-1], s=38, color=color, zorder=5)
        ax.annotate(
            format_balance(path[-1]),
            xy=(ending_year, path[-1]),
            xytext=(-8, 8),
            textcoords="offset points",
            ha="right",
            color=color,
            fontsize=13,
            fontweight="bold",
        )


# 그래프 패널 구성과 이미지 저장

def draw_case_panel(
    ax: Axes,
    start_year: int,
    case_label: str,
    qqq_path: list[float],
    deposit_path: list[float],
    y_max: float,
) -> None:
    """한 시작 연도의 QQQ·예금 잔액 경로를 패널 하나에 그린다."""
    calendar_years = [
        start_year + month / MONTHS_PER_YEAR for month in range(MONTHS + 1)
    ]
    ax.plot(
        calendar_years,
        qqq_path,
        color=QQQ_COLOR,
        linewidth=3.2,
        label="QQQ",
    )
    ax.plot(
        calendar_years,
        deposit_path,
        color=DEPOSIT_COLOR,
        linewidth=2.6,
        linestyle="--",
        label=DEPOSIT_LABEL,
    )
    ax.axhline(
        INITIAL_RESERVE,
        color=REFERENCE_COLOR,
        linewidth=1.8,
        linestyle=":",
        label="시작 준비금",
        zorder=0,
    )
    add_path_annotation(ax, qqq_path, QQQ_COLOR, start_year, 18)
    add_path_annotation(ax, deposit_path, DEPOSIT_COLOR, start_year, 34)

    ax.set_title(
        f"{start_year}년 시작 · {case_label}",
        loc="left",
        fontsize=18,
        fontweight="bold",
        color=TEXT_COLOR,
        pad=14,
    )
    ax.set_facecolor(BACKGROUND_COLOR)
    ax.set_ylim(0, y_max)
    ax.set_ylabel("잔액(억원)", color=TICK_COLOR, fontsize=13.5)
    ax.set_xlabel("연도", color=TICK_COLOR, fontsize=13.5)
    tick_step = max(1, (COMPARISON_YEARS + 9) // 10)
    tick_years = list(range(start_year, start_year + COMPARISON_YEARS + 1, tick_step))
    ending_year = start_year + COMPARISON_YEARS
    if tick_years[-1] != ending_year:
        tick_years.append(ending_year)
    ax.set_xticks(tick_years)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_eok_axis))
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(
        axis="both",
        colors=TICK_COLOR,
        labelsize=12,
        labelbottom=True,
        length=0,
    )


def save_comparison_chart(
    qqq_paths: dict[int, list[float]], deposit_path: list[float], output_path: Path
) -> None:
    """세 시작연도의 QQQ·예금 잔액을 세로 패널 그래프로 저장한다."""
    configure_korean_font()
    max_balance = max(max(path) for path in qqq_paths.values())
    y_max = max(4 * WON_PER_EOK, max_balance * 1.12)

    fig, axes = plt.subplots(len(CASES), 1, figsize=(11, 14), sharey=True)
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    fig.text(
        0.08,
        0.998,
        "대도시 연구실",
        ha="left",
        va="top",
        fontsize=9,
        fontweight="medium",
        color=FOOTNOTE_COLOR,
    )
    fig.suptitle(
        f"성장 준비금 {COMPARISON_YEARS}년 잔액 변화 | QQQ vs 연 {ANNUAL_DEPOSIT_RATE * 100:g}% 예금",
        x=0.08,
        y=0.985,
        ha="left",
        fontsize=21,
        fontweight="bold",
        color=TEXT_COLOR,
    )
    fig.text(
        0.08,
        0.943,
        f"초기 준비금 {format_balance(INITIAL_RESERVE)} · 매월 {format_balance(MONTHLY_WITHDRAWAL)} 인출 · {COMPARISON_YEARS}년",
        ha="left",
        fontsize=15.5,
        color=SECONDARY_TEXT_COLOR,
    )

    for ax, (year, case_label) in zip(axes, CASES):
        draw_case_panel(
            ax,
            start_year=year,
            case_label=case_label,
            qqq_path=qqq_paths[year],
            deposit_path=deposit_path,
            y_max=y_max,
        )

    axes[0].legend(frameon=False, loc="upper right", ncol=3, fontsize=11.5)

    fig.text(
        0.075,
        0.018,
        "QQQ 배당·분할 및 환율 반영\n※ 세금·수수료 제외",
        fontsize=13,
        linespacing=0.9,
        color=FOOTNOTE_COLOR,
    )
    fig.tight_layout(rect=(0.04, 0.045, 0.96, 0.93), h_pad=2.25)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
        facecolor=BACKGROUND_COLOR,
    )
    plt.close(fig)


# =============================================================================
# 5. 실행과 결과 표시
# =============================================================================


def prepare_detail_csv(
    detail_csv: Path | None = None,
    detail_output: Path = DEFAULT_DETAIL_OUTPUT,
) -> Path:
    """기존 CSV를 선택하거나 새 백테스트 CSV를 생성한다."""
    if detail_csv is not None:
        return detail_csv.expanduser().resolve()
    return create_detail_csv(detail_output)


def display_results(detail_csv: Path, output_path: Path) -> None:
    """실행 조건, 그래프와 다운로드 링크를 노트북에 표시한다."""
    from IPython.display import FileLink, Image, Markdown, display

    start_years = ", ".join(f"{year}년" for year in START_YEARS)
    conditions = (
        "## 실행 조건\n"
        f"- 초기 준비금: **{format_balance(INITIAL_RESERVE)}**\n"
        f"- 월 인출액: **{format_balance(MONTHLY_WITHDRAWAL)}**\n"
        f"- 예금금리: **연 {ANNUAL_DEPOSIT_RATE * 100:g}%**\n"
        f"- 비교기간: **{COMPARISON_YEARS}년**\n"
        f"- 시작연도: **{start_years}**"
    )
    display(Markdown(conditions))

    display(Markdown("## 잔액 흐름 그래프"))
    display(Image(filename=str(output_path)))
    display(Markdown("## 결과 파일 다운로드"))
    display(Markdown("**그래프 이미지(PNG)**"))
    display(FileLink(str(output_path)))
    display(Markdown("**월별 상세 데이터(CSV)**"))
    display(FileLink(str(detail_csv)))


def main(
    detail_csv: Path | None = None,
    detail_output: Path = DEFAULT_DETAIL_OUTPUT,
    output_path: Path = DEFAULT_OUTPUT,
) -> None:
    """데이터를 준비하고 잔액 경로를 계산해 그래프와 링크를 표시한다."""
    resolved_detail_csv = prepare_detail_csv(detail_csv, detail_output)
    qqq_paths = load_qqq_paths(resolved_detail_csv)
    deposit_path = calculate_deposit_path()
    save_comparison_chart(qqq_paths, deposit_path, output_path)
    display_results(resolved_detail_csv, output_path)


if __name__ == "__main__":
    main()
